# Candidate Generation: Content Baselines

This notebook builds three personalised content baselines: **text**, **entity** and a
**text/entity fusion**. It reuses the causal per-request profile references saved by
`4_Feature_Engineering` (`full_history_rows`), scores time-eligible articles by cosine
similarity, excludes every article already in the request's history, and writes the
same standard candidate format as the popularity baseline.

Protocol:

- Profiles are the pre-feedback causal history at each request. They include supplied
  history and earlier clicks, but never the current impression's labels.
- Article vectors are the frozen training-fitted representations from stage 3.
- Eligibility is `available_from <= timestamp`.
- All `full_history_rows` are removed from that request's candidates.
- Ties are broken by popularity descending, then `news_code` ascending.
- Empty/zero profiles, and any short candidate list, fall back to the popularity order.
- Evaluate Recall@M at M = 50, 100, 200 and 500 with the shared `CandidateEvaluator`.

The fused score is `0.5 * text_cosine + 0.5 * entity_cosine` when both sides have a
valid profile and article vector. If only one side is valid, that side is used alone.

In [1]:
from datetime import datetime, timezone
import json
import sys
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_all.py').is_file()
)
sys.path.insert(0, str(ROOT / '5_Models' / '5_01_Candidate_Generation'))

from cg_evaluation import CandidateEvaluator

PROCESSED = ROOT / '1_Data_Preparation' / 'processed'
FEATURES = ROOT / '4_Feature_Engineering' / 'artifacts' / 'prequential-all'
REPRESENTATIONS = ROOT / '3_Representations' / 'artifacts'
ARTIFACTS = ROOT / '5_Models' / '5_01_Candidate_Generation' / 'artifacts'
RESULTS = ROOT / 'results'

TARGET_SPLIT = 'validation'
FIT_SPLIT = 'train' if TARGET_SPLIT == 'validation' else 'train+validation'
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
MODELS = ('content_text', 'content_entity', 'content_text_entity')

if TARGET_SPLIT not in {'validation', 'test'}:
    raise ValueError('TARGET_SPLIT must be validation or test')

print('Repository:', ROOT)
print('Target split:', TARGET_SPLIT)
print('Fit split for popularity fallback:', FIT_SPLIT)
print('M values:', M_VALUES)
print('Candidate cap:', CANDIDATE_CAP)

Repository: c:\Users\Yong Rui\BT4222-Group-6-
Target split: validation
Fit split for popularity fallback: train
M values: (50, 100, 200, 500)
Candidate cap: 500


## Helper calculations

These are the profile-pooling, score-fusion and deterministic Top-500 calculations
used by all three baselines. Keeping them here makes the baseline self-contained.

- `pool_profiles` mean-pools valid history vectors and L2-normalises, preserving
  repeated references and ignoring row 0 padding.
- `fuse_scores` averages text and entity cosine when both are valid, otherwise uses
  the valid side.
- `popularity_select` is the cold-start path over a precomputed global order.
- `select_top_k` ranks by score, then popularity, then `news_code`, and never
  selects a history position.
- `CandidateWriter` buffers the standard long-format Parquet output.

In [2]:
CANDIDATE_SCHEMA = pa.schema(
    [
        ('impression_uid', pa.int64()),
        ('cutoff', pa.int16()),
        ('news_code', pa.int32()),
    ]
)


def pool_profiles(history_rows, matrix, block=10000):
    '''Mean-pool valid article vectors per request and L2-normalise.'''
    matrix = np.asarray(matrix)
    if matrix.ndim != 2:
        raise ValueError('matrix must be two-dimensional')
    n = len(history_rows)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)
    if n == 0:
        return result

    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        total = int(lengths.sum())
        if total == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        if flat.min() < 0 or flat.max() >= matrix.shape[0]:
            raise ValueError('history contains an invalid article row')
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        group_valid = groups[valid]
        vector_valid = vectors[valid]
        counts = np.bincount(group_valid, minlength=len(chunk))
        sums = np.empty((len(chunk), dim), dtype=np.float32)
        for column in range(dim):
            sums[:, column] = np.bincount(
                group_valid,
                weights=vector_valid[:, column],
                minlength=len(chunk),
            )
        present = counts > 0
        if not present.any():
            continue
        pooled = sums[present] / counts[present, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][present] = pooled

    return result


def fuse_scores(text_scores, entity_scores, text_available, entity_available):
    '''Equal-weight fusion with valid-side renormalisation.'''
    text_scores = np.asarray(text_scores, dtype=np.float32)
    entity_scores = np.asarray(entity_scores, dtype=np.float32)
    text_available = np.asarray(text_available, dtype=bool)
    entity_available = np.asarray(entity_available, dtype=bool)
    if not (
        text_scores.shape
        == entity_scores.shape
        == text_available.shape
        == entity_available.shape
    ):
        raise ValueError('score and availability shapes must match')

    fused = np.full(text_scores.shape, -np.inf, dtype=np.float32)
    both = text_available & entity_available
    fused[both] = 0.5 * text_scores[both] + 0.5 * entity_scores[both]
    only_text = text_available & ~entity_available
    fused[only_text] = text_scores[only_text]
    only_entity = entity_available & ~text_available
    fused[only_entity] = entity_scores[only_entity]
    return fused


def popularity_select(order, n_eligible, excluded=None, k=500):
    '''Take popular eligible positions from a precomputed global order.'''
    if k < 1:
        raise ValueError('k must be positive')
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible:
            continue
        if position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(
        f'only {len(selected)} popular eligible articles; cannot select {k}'
    )


def _popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    if pool.size < count:
        raise ValueError(
            f'only {pool.size} eligible articles remain; cannot fill {count}'
        )
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    '''Deterministic Top-k by score, popularity, then news_code.'''
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    if scores.ndim != 1:
        raise ValueError('scores must be one-dimensional')
    if popularity.shape != scores.shape or news_code.shape != scores.shape:
        raise ValueError('score, popularity and news_code shapes must match')
    if k < 1:
        raise ValueError('k must be positive')
    n = scores.size
    if n < k:
        raise ValueError(f'only {n} eligible articles; cannot select {k}')
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        if positions.min() < 0 or positions.max() >= n:
            raise ValueError('excluded position outside the eligible array')
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return _popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort((news_code[ties], -popularity[ties]))
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = _popularity_fill(
            remaining, popularity, news_code, k - selected.size
        )
        selected = np.r_[selected, fill]

    return selected


class CandidateWriter:
    '''Buffered writer for the standard long-format candidate table.'''

    def __init__(self, path, batch_rows=2_000_000):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.batch_rows = batch_rows
        self.frames = []
        self.pending = 0
        self.rows = 0
        self.writer = pq.ParquetWriter(
            self.path, CANDIDATE_SCHEMA, compression='zstd'
        )

    def add(self, impression_uid, cutoff, news_code):
        frame = pa.table(
            {
                'impression_uid': pa.array(impression_uid, type=pa.int64()),
                'cutoff': pa.array(cutoff, type=pa.int16()),
                'news_code': pa.array(news_code, type=pa.int32()),
            },
            schema=CANDIDATE_SCHEMA,
        )
        self.frames.append(frame)
        self.pending += frame.num_rows
        self.rows += frame.num_rows
        if self.pending >= self.batch_rows:
            self.flush()

    def flush(self):
        if not self.frames:
            return
        table = pa.concat_tables(self.frames)
        self.writer.write_table(table)
        self.frames = []
        self.pending = 0

    def close(self):
        self.flush()
        self.writer.close()

## 1. Load and validate the causal request profiles

Every request must have exactly one profile row, with the same user and timestamp as
the impression table. The profile rows are the full causal history used by stage 4;
they are the request-time state before current feedback.

In [3]:
profile_parts = sorted((FEATURES / f'request_profiles_{TARGET_SPLIT}').glob('*.parquet'))
if not profile_parts:
    raise FileNotFoundError(f'No request profiles under {FEATURES}')

profiles = pl.read_parquet(
    profile_parts,
    columns=['impression_uid', 'user_code', 'timestamp', 'full_history_rows'],
)
impressions = pl.read_parquet(
    PROCESSED / f'impressions_{TARGET_SPLIT}.parquet',
    columns=['impression_uid', 'user_code', 'timestamp'],
)

if profiles.height != impressions.height:
    raise ValueError(
        f'Profile rows ({profiles.height:,}) != impressions ({impressions.height:,})'
    )
if profiles['impression_uid'].n_unique() != profiles.height:
    raise ValueError('Request profiles contain duplicate impression_uid values')
if profiles['full_history_rows'].null_count():
    raise ValueError('Request profiles contain null full_history_rows')

profile_check = profiles.select(
    'impression_uid',
    pl.col('user_code').alias('profile_user_code'),
    pl.col('timestamp').alias('profile_timestamp'),
)
check = impressions.join(profile_check, on='impression_uid', how='full', coalesce=True)
if check.height != impressions.height:
    raise ValueError('Profile/impression join changed the row count')
if check['profile_timestamp'].null_count():
    raise ValueError('Some impressions have no request profile')
if not (check['timestamp'] == check['profile_timestamp']).all():
    raise ValueError('Profile timestamps do not match impression timestamps')
if not (check['user_code'] == check['profile_user_code']).all():
    raise ValueError('Profile users do not match impression users')

index = pl.read_parquet(
    REPRESENTATIONS / 'article_index.parquet',
    columns=['article_row', 'news_code'],
)
n_articles = index.height + 1
row_stats = profiles.select(
    pl.col('full_history_rows').list.len().alias('history_length'),
    pl.col('full_history_rows').list.min().alias('min_row'),
    pl.col('full_history_rows').list.max().alias('max_row'),
)
if row_stats['history_length'].null_count():
    raise ValueError('Invalid history length in request profiles')
if row_stats['max_row'].max() >= n_articles:
    raise ValueError('Request profile references an article row outside the catalogue')
if row_stats['min_row'].drop_nulls().min() < 0:
    raise ValueError('Request profile contains a negative article row')

profiles = profiles.sort('timestamp')
history_rows = profiles['full_history_rows'].to_list()
timestamps = profiles['timestamp'].to_numpy()
impression_uids = profiles['impression_uid'].to_numpy()
empty_profiles = int((row_stats['history_length'] == 0).sum())

print(f'Requests: {profiles.height:,}')
print(f'Unique timestamps: {profiles["timestamp"].n_unique():,}')
print(f'Catalogue articles: {n_articles - 1:,}')
print(f'Requests with empty causal history: {empty_profiles:,}')
print(f'Mean history length: {row_stats["history_length"].mean():.2f}')

Requests: 334,913
Unique timestamps: 54,728
Catalogue articles: 104,151
Requests with empty causal history: 1,021
Mean history length: 40.32


## 2. Load article vectors, availability and the popularity fallback

Articles are ordered by `available_from` so each timestamp is a prefix. The
popularity order is the same prior-exposure order used by the checked-in popularity
artifact; it is used only for cold starts and short content lists.

In [4]:
text = np.load(REPRESENTATIONS / 'text64.npy', mmap_mode='r')
entity = np.load(REPRESENTATIONS / 'article_entity100.npy', mmap_mode='r')
if text.shape[0] != n_articles or entity.shape[0] != n_articles:
    raise ValueError('Article arrays do not align with the catalogue')

availability = pl.read_parquet(
    REPRESENTATIONS / 'article_availability.parquet',
    columns=['news_code', 'available_from'],
)
catalogue = index.join(availability, on='news_code', how='left')
if catalogue.height != index.height:
    raise ValueError('Availability join changed the catalogue row count')
if catalogue['available_from'].null_count():
    raise ValueError('Missing available_from for some catalogue articles')

counts = pl.read_parquet(
    PROCESSED / 'news_catalogue.parquet',
    columns=[
        'news_code',
        'exposures_train',
        'exposures_validation',
    ],
)
prior = pl.col('exposures_train')
if FIT_SPLIT == 'train+validation':
    prior = prior + pl.col('exposures_validation')
popularity = counts.select('news_code', prior.alias('popularity_score'))
catalogue = catalogue.join(popularity, on='news_code', how='left')
if catalogue['popularity_score'].null_count():
    raise ValueError('Missing popularity score for some catalogue articles')

ordered = catalogue.sort(['available_from', 'news_code'])
article_rows = ordered['article_row'].to_numpy().astype(np.int64)
news_sorted = ordered['news_code'].to_numpy().astype(np.int64)
available_sorted = ordered['available_from'].to_numpy().astype('datetime64[ns]')
popularity_sorted = ordered['popularity_score'].to_numpy().astype(np.int64)
text_sorted = np.asarray(text[article_rows], dtype=np.float32)
entity_sorted = np.asarray(entity[article_rows], dtype=np.float32)
text_article_valid = np.linalg.norm(text_sorted, axis=1) > 0
entity_article_valid = np.linalg.norm(entity_sorted, axis=1) > 0

pos_in_sorted = np.empty(n_articles, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(len(article_rows), dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

if not np.all(available_sorted[1:] >= available_sorted[:-1]):
    raise ValueError('Availability order is not non-decreasing')

print(f'Text article vectors: {int(text_article_valid.sum()):,} / {len(article_rows):,}')
print(f'Entity article vectors: {int(entity_article_valid.sum()):,} / {len(article_rows):,}')
print(f'Popularity scores: {len(popularity_sorted):,}')

Text article vectors: 104,151 / 104,151
Entity article vectors: 91,452 / 104,151
Popularity scores: 104,151


## 3. Pool each request's causal history

Pooling uses the same valid-vector mean and L2 normalisation as the profile code in
stage 3/4. Repeated history references are preserved. Empty or all-zero histories
produce a zero profile, which triggers the popularity fallback.

In [5]:
text_profiles = pool_profiles(history_rows, text)
entity_profiles = pool_profiles(history_rows, entity)
text_profile_valid = np.linalg.norm(text_profiles, axis=1) > 0
entity_profile_valid = np.linalg.norm(entity_profiles, axis=1) > 0

print(f'Valid text profiles: {int(text_profile_valid.sum()):,} / {len(text_profiles):,}')
print(f'Valid entity profiles: {int(entity_profile_valid.sum()):,} / {len(entity_profiles):,}')
print(f'No text profile: {int((~text_profile_valid).sum()):,}')
print(f'No entity profile: {int((~entity_profile_valid).sum()):,}')

Valid text profiles: 333,892 / 334,913
Valid entity profiles: 333,362 / 334,913
No text profile: 1,021
No entity profile: 1,551


## 4. Score, exclude history and write Top-500 candidates

Each timestamp is processed as a batch. Candidate positions are mapped back to
`news_code` and written in the shared long format with `cutoff = 1..500`.

In [6]:
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
paths = {
    model: ARTIFACTS / model / f'recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet'
    for model in MODELS
}
writers = {model: CandidateWriter(path) for model, path in paths.items()}
starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]

try:
    for batch_no, (start, end) in enumerate(zip(starts, ends), 1):
        timestamp = timestamps[start]
        n_eligible = int(np.searchsorted(available_sorted, timestamp, side='right'))
        if n_eligible < CANDIDATE_CAP:
            raise ValueError(
                f'Only {n_eligible} eligible articles at {timestamp}'
            )

        popularity_batch = popularity_sorted[:n_eligible]
        news_batch = news_sorted[:n_eligible]
        text_batch = text_sorted[:n_eligible]
        entity_batch = entity_sorted[:n_eligible]

        text_scores = text_profiles[start:end] @ text_batch.T
        entity_scores = entity_profiles[start:end] @ entity_batch.T

        text_available = (
            text_profile_valid[start:end, None] & text_article_valid[None, :n_eligible]
        )
        entity_available = (
            entity_profile_valid[start:end, None]
            & entity_article_valid[None, :n_eligible]
        )
        text_scores[~text_available] = -np.inf
        entity_scores[~entity_available] = -np.inf
        fused_scores = fuse_scores(
            text_scores, entity_scores, text_available, entity_available
        )

        batch_size = end - start
        selected_batch = {
            'content_text': np.empty((batch_size, CANDIDATE_CAP), dtype=np.int64),
            'content_entity': np.empty((batch_size, CANDIDATE_CAP), dtype=np.int64),
            'content_text_entity': np.empty(
                (batch_size, CANDIDATE_CAP), dtype=np.int64
            ),
        }
        for local, profile_row in enumerate(range(start, end)):
            rows = history_rows[profile_row]
            excluded = (
                pos_in_sorted[np.asarray(rows, dtype=np.int64)]
                if rows
                else np.empty(0, dtype=np.int64)
            )
            excluded = excluded[excluded < n_eligible]

            # Cold-start requests rank eligible articles by popularity only.
            text_fallback = (
                None if text_profile_valid[profile_row] else global_popularity_order
            )
            entity_fallback = (
                None
                if entity_profile_valid[profile_row]
                else global_popularity_order
            )
            fused_fallback = (
                None
                if (text_profile_valid[profile_row] or entity_profile_valid[profile_row])
                else global_popularity_order
            )

            selected_batch['content_text'][local] = select_top_k(
                text_scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=text_fallback,
            )
            selected_batch['content_entity'][local] = select_top_k(
                entity_scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=entity_fallback,
            )
            selected_batch['content_text_entity'][local] = select_top_k(
                fused_scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=fused_fallback,
            )

        uids = impression_uids[start:end]
        expanded_uid = np.repeat(uids, CANDIDATE_CAP)
        expanded_cutoff = np.tile(cutoff_values, batch_size)
        for model in MODELS:
            writer = writers[model]
            writer.add(
                expanded_uid,
                expanded_cutoff,
                news_batch[selected_batch[model].reshape(-1)],
            )

        if batch_no % 5000 == 0 or batch_no == len(starts):
            print(
                f'Scored {batch_no:,} / {len(starts):,} timestamp batches; '
                f'{writer.rows:,} rows per model',
                flush=True,
            )
finally:
    for writer in writers.values():
        writer.close()

expected_rows = profiles.height * CANDIDATE_CAP
for model, path in paths.items():
    actual = pl.scan_parquet(path).select(pl.len()).collect().item()
    if actual != expected_rows:
        raise ValueError(
            f'{model}: expected {expected_rows:,} rows, found {actual:,}'
        )
    print(f'{model}: {actual:,} candidate rows -> {path}')

Scored 5,000 / 54,728 timestamp batches; 23,876,000 rows per model
Scored 10,000 / 54,728 timestamp batches; 48,477,500 rows per model
Scored 15,000 / 54,728 timestamp batches; 71,848,500 rows per model
Scored 20,000 / 54,728 timestamp batches; 95,868,500 rows per model
Scored 25,000 / 54,728 timestamp batches; 115,953,500 rows per model
Scored 30,000 / 54,728 timestamp batches; 132,127,000 rows per model
Scored 35,000 / 54,728 timestamp batches; 143,725,500 rows per model
Scored 40,000 / 54,728 timestamp batches; 151,681,000 rows per model
Scored 45,000 / 54,728 timestamp batches; 158,234,000 rows per model
Scored 50,000 / 54,728 timestamp batches; 163,728,500 rows per model
Scored 54,728 / 54,728 timestamp batches; 167,456,500 rows per model
content_text: 167,456,500 candidate rows -> c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_text\recommendations_validation_top500.parquet
content_entity: 167,456,500 candidate rows -> c:\Users\Yong Rui\BT42

## 5. Save manifests

Content baselines have no fitted click model; the manifest records the fixed
representation, the profile source, the exclusion policy and the fallback order.

In [7]:
created_at = datetime.now(timezone.utc).isoformat()
score_text = 'cosine(text64 request profile, text64 article vector)'
score_entity = 'cosine(article_entity100 request profile, article_entity100 article vector)'
score_fused = (
    '0.5 * text cosine + 0.5 * entity cosine when both valid; '
    'valid side alone when one is missing'
)
for model in MODELS:
    model_dir = ARTIFACTS / model
    score = {
        'content_text': score_text,
        'content_entity': score_entity,
        'content_text_entity': score_fused,
    }[model]
    manifest = {
        'model': model,
        'score': score,
        'profile_source': f'4_Feature_Engineering/artifacts/prequential-all/request_profiles_{TARGET_SPLIT}',
        'profile_column': 'full_history_rows',
        'profile_pooling': 'valid-vector mean, L2 normalised; no decay',
        'history_policy': 'exclude every article in full_history_rows',
        'fallback': 'popularity order (prior exposures) for cold or short lists',
        'fit_split': FIT_SPLIT,
        'target_split': TARGET_SPLIT,
        'candidate_cap': CANDIDATE_CAP,
        'candidate_format': list(('impression_uid', 'cutoff', 'news_code')),
        'eligibility': 'available_from <= timestamp',
        'catalogue_rows': index.height,
        'target_impressions': profiles.height,
        'unique_timestamps': int(len(starts)),
        'recommendation_rows': expected_rows,
        'recommendations_file': paths[model].name,
        'created_at': created_at,
    }
    (model_dir / f'manifest_{TARGET_SPLIT}.json').write_text(
        json.dumps(manifest, indent=2) + '\n'
    )
    print('Manifest saved:', model_dir / f'manifest_{TARGET_SPLIT}.json')

Manifest saved: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_text\manifest_validation.json
Manifest saved: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_entity\manifest_validation.json
Manifest saved: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_text_entity\manifest_validation.json


## 6. Evaluate Recall@M

The shared evaluator checks eligibility, unique candidates, complete cutoffs and full
impression coverage before computing recall.

In [8]:
evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
evaluations = []
for model in MODELS:
    evaluation = evaluator.evaluate(
        paths[model],
        model_name=model,
        m_values=M_VALUES,
    )
    evaluation.write_csv(RESULTS / f'candidate_generation_{model}_{TARGET_SPLIT}.csv')
    evaluations.append(evaluation)

comparison = pl.concat(evaluations)
popularity_path = RESULTS / f'candidate_generation_popularity_{TARGET_SPLIT}.csv'
if popularity_path.exists():
    comparison = pl.concat([pl.read_csv(popularity_path), comparison])
comparison.write_csv(
    RESULTS / f'candidate_generation_comparison_{TARGET_SPLIT}.csv'
)
comparison.sort(['m', 'model'])

model,split,m,recall_pooled,recall_impression_mean,hit_rate,impressions,relevant_clicks,retrieved_relevant,mean_candidates,mean_eligible_pool,eligible_click_coverage
str,str,i64,f64,f64,f64,i64,i64,i64,f64,f64,f64
"""content_entity""","""validation""",50,0.001626,0.001717,0.002475,334913,514658,837,50.0,99816.190673,1.0
"""content_text""","""validation""",50,0.001772,0.001963,0.002717,334913,514658,912,50.0,99816.190673,1.0
"""content_text_entity""","""validation""",50,0.00328,0.003495,0.005001,334913,514658,1688,50.0,99816.190673,1.0
"""popularity""","""validation""",50,0.0028,0.002252,0.004258,334913,514658,1441,50.0,99816.190673,1.0
"""content_entity""","""validation""",100,0.003087,0.00325,0.00467,334913,514658,1589,100.0,99816.190673,1.0
…,…,…,…,…,…,…,…,…,…,…,…
"""popularity""","""validation""",200,0.046281,0.044201,0.067412,334913,514658,23819,200.0,99816.190673,1.0
"""content_entity""","""validation""",500,0.011563,0.012101,0.017085,334913,514658,5951,500.0,99816.190673,1.0
"""content_text""","""validation""",500,0.017598,0.018465,0.02658,334913,514658,9057,500.0,99816.190673,1.0


## 7. What these baselines do and do not do

- The profiles are **causal and pre-feedback** for each request, but they consume
  earlier validation clicks. The checked-in popularity baseline is frozen on its fit
  split, so this comparison is prequential content versus frozen popularity.
- Excluding the full causal history deliberately removes repeat-click opportunities
  from the candidate set. The logged slates do contain a small amount of history
  overlap, so recall on repeat clicks is not measured here.
- Text vectors are fitted on the training catalogue; entity vectors are frozen
  pretrained vectors. Both are shared across splits.
- Users with no valid profile, and any list with fewer than 500 finite scores, fall
  back to popularity. This keeps the candidate format complete and cold-start safe.
- `available_from` is the project-wide observed-availability proxy, not a verified
  publication timestamp.